# Task 3 - Jointly Trained Soft Mixture-of-Experts Restoration (Colab runner)

A gating network gives every input a continuous weight for four branches - identity, salt-and-pepper expert, blur expert, occlusion expert - and the output is the weighted sum of the branch outputs:

`x_hat = w0 * x + w1 * A_salt(x) + w2 * A_blur(x) + w3 * A_occlusion(x)` with `w = softmax(G(x) / tau)`.

Training has two stages: a short **warm-up** in which only the gate learns (experts frozen), then **joint fine-tuning** of everything with smaller learning rates. The design decisions are explained in `docs/task3_notes.md`.

> ### Task 2 must be trained first
> Nothing here starts from random weights: the gate is the **trained Task 2 classifier** and the three experts are the **trained Task 2 specialists**, loaded from `MyDrive/GenAI_A1/checkpoints/task2/classifier/best.pt` and `.../checkpoints/task2/specialist_{salt,blur,occlusion}/best.pt`. Run `notebooks/task2_colab.ipynb` to the end before this notebook; cell 3 below checks the four files and stops if any is missing. The full comparison in cell 8 also needs the Task 1 and Task 2 **evaluations** to have run.

Run the cells **from top to bottom** on a **T4 GPU** (Runtime -> Change runtime type -> T4 GPU). Every run time below is an **estimate** for a T4; the real time scales with the size of the Task 2 models you trained, because each forward pass runs the gate and all three experts.

**If Colab disconnects:** reconnect, run cells 1 and 2 again, then re-run the cell that was interrupted. Everything that matters is on Google Drive and resumes by itself: the Optuna study keeps its finished trials and runs only the remaining ones, training continues from the last finished epoch (also across the warm-up -> joint boundary) and re-attaches to the same W&B run, a training run that already finished returns immediately, and the evaluation, comparison and export cells simply recompute their outputs.

## 1. Get the code

In [ ]:
# 1. code
REPO_URL = 'https://github.com/fsdev87/generative-ai-A1.git'
import os
if os.path.exists('/content/repo'):
    !git -C /content/repo pull
else:
    !git clone {REPO_URL} /content/repo
%cd /content/repo

## 2. Setup: Drive, packages, data cache, W&B, environment variables
Mounts Drive, installs the requirements, makes sure the 128x128 Pets cache exists and points `CKPT_DIR`, `OPTUNA_DIR`, `ONNX_DIR` and `OUTPUT_DIR` at Drive. Needs the Colab secret `WANDB_API_KEY`. Estimated time: 1-3 min (longer in the very first session, which downloads and caches the dataset).

In [ ]:
# 2. setup: Drive, packages, data cache, W&B, env vars
from src.common.colab import setup
setup(fs2k=False)   # True for Task 4

## 3. Check that the Task 2 models exist (a few seconds)
Prints the four checkpoints the MoE is built from and stops with an error if one is missing. The experts are built from each checkpoint's own `model_config`, so whatever architecture Task 2 selected (for example a 16x16 latent) is used as it is.

In [ ]:
from src.common.paths import get_dir
from src.task3.model import missing_task2_checkpoints, task2_checkpoint_paths

for name, path in task2_checkpoint_paths(get_dir('CKPT_DIR')).items():
    print(f"{name:10s} {'OK     ' if path.exists() else 'MISSING'} {path}")
missing = missing_task2_checkpoints(get_dir('CKPT_DIR'))
assert not missing, 'Train Task 2 first: notebooks/task2_colab.ipynb'
print('\nAll Task 2 checkpoints found.')

## 4. Quick check (optional, estimate approx. 1-2 min)
A tiny end-to-end run on synthetic data: both stages, one epoch each, W&B disabled, everything written to a separate `smoke` folder so it can never touch the real results. If this fails, fix the problem before spending GPU time.

In [ ]:
!python -m src.task3.train --smoke

## 5. Optuna hyperparameter search (estimate approx. 1-1.5 h)
Study `task3_moe`, W&B group `task3-optuna`, one run per trial. Searched, as the brief requires: the **joint fine-tuning learning rate**, the **temperature tau**, the **classification weight** `lambda_ce`, the **balance weight** `lambda_balance` and the **reconstruction weighting** `alpha` (`lambda_l1 = alpha`, `lambda_ssim = 1 - alpha`). Trial 0 is the brief's starting point (0.8 / 0.2 / 0.1 / 0.01, tau = 1) so every other trial is measured against it.

Each trial trains a short schedule (1 warm-up + 4 joint epochs, estimate approx. 4-6 min) and is stopped early when it is clearly worse than the others (median pruner) **or when the routing collapses** on the validation set - a branch receiving less than 5 % of the class-balanced weight, or one branch taking more than 50 % of the weight on inputs of the other three classes. The pruning reason is stored with the trial and appears in `trials.csv`.

The session stops after 20 finished trials, or as soon as 90 minutes have passed (no new trial is started after that). **After a disconnect just run this cell again**: finished trials live in the study database on Drive and only the remaining ones run. Outputs: `OUTPUT_DIR/task3/optuna/task3_moe/` (history, importances, parallel coordinates, slice plots, `trials.csv`, `summary.json`) and the selected configuration `OUTPUT_DIR/task3/best_config.yaml`.

In [ ]:
!python -m src.task3.optuna_search --n-trials 20 --timeout-min 90

## 6. Final training with the best configuration (estimate approx. 20-35 min)
2 warm-up epochs (gate only, experts frozen) followed by up to 20 joint epochs, with early stopping after 6 joint epochs without improvement. W&B group `task3-final` logs every loss component (L1, 1 - SSIM, cross-entropy, balance), the validation metrics per corruption type, the **4x4 routing heatmap** every epoch and sample images of fixed validation entries.

Checkpoints: `CKPT_DIR/task3/moe/last.pt` (written every epoch, holds the full training state including the stage) and `best.pt` (best validation score, also uploaded as a W&B artifact). Re-running this cell after a disconnect continues from the last finished epoch; if the run already finished it prints that and returns.

In [ ]:
!python -m src.task3.train --config $OUTPUT_DIR/task3/best_config.yaml --resume

## 7. Test-set evaluation and gating analysis (estimate approx. 5-8 min)
Runs `best.pt` in float32 (the precision of the exported ONNX model) over all 36,690 test entries and writes everything the report needs to `OUTPUT_DIR/task3/`:

- `eval/test_records.csv` - one row per entry with PSNR/SSIM/MSE, the four routing weights, the dominant branch and the normalised routing entropy; `eval/summary.json`.
- `tables/test_by_*` - restoration quality per corruption type and severity (CSV + LaTeX).
- **the gating analysis the brief asks for**: `tables/routing_by_type_level` (average expert weights for every true corruption type *and* severity level) with the matching heatmap, `figures/weight_distributions.png` (per-class distribution of each branch weight), `figures/gating_dominant.png` and `figures/gating_spread.png` (examples where one expert dominates and where the weights are spread, each with every branch output and its weight), and `tables/expert_activity.csv/.tex` + `figures/routing_entropy.png`, which quantify whether an expert became **inactive** or **dominates unrelated inputs**.
- `figures/examples_*.png` and `figures/failures*.png` - Target | Input | Output | |Error| rows in the same style as Tasks 1 and 2, including failure cases and the worst mis-routed entries.

In [ ]:
!python -m src.task3.evaluate

## 8. Comparison with Tasks 1 and 2 (estimate < 1 min)
Builds one table per corruption type x severity comparing **universal autoencoder** (Task 1) vs **hard routing with the true label** (Task 2 oracle) vs **hard routing with the classifier** (Task 2 predicted) vs **soft MoE** (Task 3), reading the per-entry CSVs of the other tasks. Because all systems are evaluated on the same manifest in the same order, it also reports paired per-entry gains and win rates, split by whether the Task 2 classifier was right - which is where soft routing is expected to help.

Inputs that do not exist yet are skipped with a message, so this cell is safe to run early; for the complete table run the Task 1 and Task 2 evaluation notebooks first. Outputs: `OUTPUT_DIR/task3/tables/comparison_*` and `figures/comparison_{psnr,ssim}.png`.

In [ ]:
!python -m src.task3.compare

## 9. Extra experiment: images with two corruptions (estimate approx. 1-2 min)
**Beyond the brief's corruption definitions, and labelled as such in the report.** The brief motivates soft routing with images that contain more than one corruption, but its test manifest has exactly one corruption per entry. `manifests/pets_mixed_manifest.json` (committed to the repository, built once by `python -m src.task3.make_mixed_manifest`) holds 300 fixed test images x {salt+blur, blur+occlusion, salt+occlusion}, each component at its fixed medium test severity.

The cell compares the soft MoE with the same MoE forced to argmax routing, with Task 2 hard routing and with the Task 1 universal model, and reports the mean routing weights per combination plus how often the two largest weights are exactly the two corruptions present. Outputs: `OUTPUT_DIR/task3/mixed/`.

In [ ]:
!python -m src.task3.evaluate_mixed

## 10. ONNX export for the application (estimate approx. 1 min)
Exports the **complete** soft-MoE inference pipeline as one graph - gate, temperature, softmax, identity branch, the three experts and the weighted sum - to `ONNX_DIR/moe.onnx` with input `input` and outputs `output`, `weights` and `branch_outputs`, as the application expects. It then compares ONNX Runtime with PyTorch on 40 real test images (4 per corruption type and severity) and **fails loudly** if any output differs by more than 1e-4, checks batch sizes 1 and 3 for the dynamic batch axis, and writes the model card `moe.json` (tau, model configs, test metrics, routing summary, parity result, W&B run).

In [ ]:
!python -m src.task3.export_onnx

## Where the results are (on Google Drive, `MyDrive/GenAI_A1/`)
- `checkpoints/task3/moe/best.pt` - the final model (`last.pt` holds the full training state for resuming)
- `optuna/task3_moe.db` - the Optuna study; `outputs/task3/optuna/task3_moe/` - its report for the paper
- `outputs/task3/best_config.yaml` - the selected hyperparameters (copy it into the repository as `configs/task3_best.yaml` so the run is reproducible)
- `outputs/task3/eval/`, `outputs/task3/tables/`, `outputs/task3/figures/`, `outputs/task3/mixed/` - all results, tables and figures for the report
- `onnx/moe.onnx` + `onnx/moe.json` - the model and its card for the application